# A/B-тест: влияние геймификации на учебную активность и удержание

**Проект:** оценка новой механики достижений за регулярное решение домашних заданий  
**Период событий в выгрузке:** 4–24 мая 2026 года (21 день)  
**Группы:** A - контроль, B - новый интерфейс с геймификацией

## Цель анализа

Проверить, меняет ли новая механика поведение учеников: повышает ли возвраты на 7-й и 14-й день, увеличивает ли долю выполненных домашних заданий и ускоряет ли первое достижение ценности (Time to Value, TTV).



## Важные оговорки по данным

1. В событиях нет отдельного признака **полного завершения курса или модуля**. Поэтому `completed_hw / homework_started` - это пользовательский коэффициент завершения начатых домашних заданий, а не Completion Rate всего курса. Дополнительно считаем долю пользователей, у которых было хотя бы одно завершённое домашнее задание.
2. `return_visit_d7` и `return_visit_d14` - готовые бинарные маркеры. В исходной выгрузке все события одного типа имеют одинаковый timestamp, поэтому самостоятельно проверить дату реального возврата по этим временным меткам нельзя.
3. Для TTV в исходном анализе учитываются только пользователи, у которых нашлось событие `homework_completed`. Поэтому TTV - дополнительная метрика **среди завершивших хотя бы одно задание**, а не среди всей выборки.
4. Для воспроизводимого запуска в файлы ноутбука надо положить `ab_test_users_groups.csv` и `ab_test_events.csv`.

---

## 1. План анализа и гипотезы

### Основные гипотезы

Для каждой метрики сравниваем тестовую группу B с контрольной группой A.

- **H₀:** метрика в группах A и B одинакова; наблюдаемая разница объясняется случайностью.
- **H₁:** метрика в группах A и B различается.

Для метрик, где больше - лучше (Retention, доля завершённых заданий), положительная разница B - A означает улучшение. Для TTV меньшее значение лучше, поэтому отрицательная разница B - A означает ускорение первого завершения.

### Уровень значимости

Используем двусторонний уровень значимости $(\alpha = 0.05)$ и 95%-е доверительные интервалы. Для нескольких проверяемых метрик дополнительно применяем поправку Холма. Это снижает вероятность получить случайно значимый результат из-за большого числа сравнений.

### Что считаем основными метриками в этой выгрузке

- Retention D7 и D14 - доля пользователей с соответствующим маркером возврата.
- Доля пользователей с хотя бы одним `homework_completed` - конверсия до первого завершённого задания.
- Средний пользовательский `completion_rate = completed_hw / hw_started` - отношение числа завершённых домашних заданий к числу начатых на уровне пользователя.
- TTV - время от первого `login` до первого `homework_completed`, только среди пользователей с обоими событиями.


## 2. Подготовка окружения и загрузка данных

Сначала импортируем библиотеки и читаем два CSV. Функция ниже проверяет наличие файлов в текущей папке ноутбука и, если ноутбук запущен в Google Colab, дополнительно проверяет `/content`.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ALPHA = 0.05
N_BOOT = 2_000
RANDOM_STATE = 42
OUTLIER_THRESHOLD = 25
RETURN_EVENTS = {"return_visit_d7", "return_visit_d14"}

possible_dirs = [Path.cwd(), Path("/content")]
DATA_DIR = next(
    (folder for folder in possible_dirs
     if (folder / "ab_test_users_groups.csv").exists()
     and (folder / "ab_test_events.csv").exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Не найдены ab_test_users_groups.csv и ab_test_events.csv. "
        "Положите оба CSV в одну папку с ноутбуком и запустите ячейки ещё раз."
    )

users_path = DATA_DIR / "ab_test_users_groups.csv"
events_path = DATA_DIR / "ab_test_events.csv"

df_users_raw = pd.read_csv(users_path)
df_events_raw = pd.read_csv(events_path)

print("Папка с данными:", DATA_DIR.resolve())
print("Таблица пользователей:", df_users_raw.shape)
print("Таблица событий:", df_events_raw.shape)

Папка с данными: /content
Таблица пользователей: (6008, 4)
Таблица событий: (179330, 3)


### Первичный осмотр

Проверяем размер таблиц, поля, типы событий и период логов.

In [ ]:
display(df_users_raw.head())
display(df_events_raw.head())

print("Поля пользователей:", df_users_raw.columns.tolist())
print("Поля событий:", df_events_raw.columns.tolist())
print("Типы событий:")
display(df_events_raw["event_type"].value_counts(dropna=False).to_frame("events"))

,user_id,group,signup_date,grade
0,U300000,B,2026-03-09,5
1,U300001,B,2025-09-07,9
2,U300002,B,2026-01-11,9
3,U300003,A,2025-06-07,5
4,U300004,B,2025-12-25,11


,user_id,timestamp,event_type
0,U300281,2026-05-04 00:06:44,homework_completed
1,U300733,2026-05-04 00:19:56,homework_completed
2,U304499,2026-05-04 00:22:07,login
3,U304266,2026-05-04 00:23:42,homework_started
4,U304266,2026-05-04 00:23:42,homework_started


Поля пользователей: ['user_id', 'group', 'signup_date', 'grade']
Поля событий: ['user_id', 'timestamp', 'event_type']
Типы событий:


,events
event_type,
login,75552
homework_started,68317
homework_completed,28995
return_visit_d7,4519
return_visit_d14,1947


In [ ]:
# Приводим даты к datetime. Некорректные значения превратятся в NaT и будут учтены отдельно.
df_users_raw["signup_date"] = pd.to_datetime(df_users_raw["signup_date"], errors="coerce")
df_events_raw["timestamp"] = pd.to_datetime(df_events_raw["timestamp"], errors="coerce")

print("Период событий:", df_events_raw["timestamp"].min(), "—", df_events_raw["timestamp"].max())
print("Некорректных timestamp:", df_events_raw["timestamp"].isna().sum())
print("Пропусков в таблице пользователей:")
display(df_users_raw.isna().sum().to_frame("missing"))
print("Пропусков в таблице событий:")
display(df_events_raw.isna().sum().to_frame("missing"))

Период событий: 2026-05-04 00:06:44 — 2026-05-24 23:55:43
Некорректных timestamp: 0
Пропусков в таблице пользователей:


,missing
user_id,0
group,0
signup_date,0
grade,0


Пропусков в таблице событий:


,missing
user_id,0
timestamp,0
event_type,0


## 3. Проверка качества и очистка данных

### 3.1. Конфликтующие назначения пользователей

Один пользователь должен принадлежать ровно к одной группе. В исходной выгрузке 8 `user_id` встречаются дважды - по одной записи в A и B. Нельзя произвольно оставить одну из групп: это будет противоречить независимости групп. Поэтому таких пользователей исключаем целиком из анализа.

In [ ]:
required_user_cols = {"user_id", "group", "signup_date", "grade"}
required_event_cols = {"user_id", "timestamp", "event_type"}
assert required_user_cols.issubset(df_users_raw.columns), "В users отсутствуют обязательные поля"
assert required_event_cols.issubset(df_events_raw.columns), "В events отсутствуют обязательные поля"

# События с отсутствующим user_id / event_type / timestamp нельзя корректно использовать.
invalid_event_rows = df_events_raw[["user_id", "timestamp", "event_type"]].isna().any(axis=1)
print("Строк событий с пропусками в обязательных полях:", invalid_event_rows.sum())
df_events_valid = df_events_raw.loc[~invalid_event_rows].copy()

# Ищем все записи пользователей, которые указаны более одного раза.
duplicate_user_rows = df_users_raw[df_users_raw.duplicated("user_id", keep=False)].copy()
duplicate_user_ids = duplicate_user_rows["user_id"].unique()

print("Количество строк в таблице назначения, относящихся к дублям:", len(duplicate_user_rows))
print("Количество конфликтующих user_id:", len(duplicate_user_ids))
display(duplicate_user_rows.sort_values("user_id"))

# Исключаем все записи конфликтующих user_id.
df_users_unique = df_users_raw[~df_users_raw["user_id"].isin(duplicate_user_ids)].copy()
assert df_users_unique["user_id"].is_unique

print("Пользователей после устранения конфликтующих назначений:", len(df_users_unique))
display(df_users_unique["group"].value_counts().reindex(["A", "B"]))

Строк событий с пропусками в обязательных полях: 0
Количество строк в таблице назначения, относящихся к дублям: 16
Количество конфликтующих user_id: 8


,user_id,group,signup_date,grade
357,U300357,A,2026-03-19,9
6000,U300357,B,2026-03-19,9
460,U300460,B,2025-08-12,6
6001,U300460,A,2025-08-12,6
920,U300920,B,2025-04-09,10
6002,U300920,A,2025-04-09,10
1014,U301014,A,2026-03-28,10
6003,U301014,B,2026-03-28,10
1087,U301087,B,2025-06-27,10
6004,U301087,A,2025-06-27,10


Пользователей после устранения конфликтующих назначений: 5992


,count
group,
A,2997
B,2995


### 3.2. Дубликаты событий и объединение таблиц

Проверяем полные дубликаты по трём полям: пользователь, timestamp и тип события. Удаляем только полные совпадения. Разные типы событий одного пользователя в одну минуту не считаем дублями автоматически.

После этого присоединяем группу и класс к каждому событию. `validate="many_to_one"` проверяет, что в таблице пользователей после очистки действительно одна строка на пользователя.

In [ ]:
event_duplicate_mask = df_events_valid.duplicated(
    subset=["user_id", "timestamp", "event_type"], keep="first"
)
print("Полных дубликатов событий:", event_duplicate_mask.sum())

df_events_unique = df_events_valid.loc[~event_duplicate_mask].copy()

df_merged_before_outliers = df_events_unique.merge(
    df_users_unique,
    on="user_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

unmatched_events = (df_merged_before_outliers["_merge"] != "both").sum()
print("Событий без записи о пользователе:", unmatched_events)

# Оставляем только события пользователей с однозначным назначением в эксперименте.
df_merged_before_outliers = (
    df_merged_before_outliers
    .loc[df_merged_before_outliers["_merge"] == "both"]
    .drop(columns="_merge")
    .copy()
)

print("Размер объединённой таблицы:", df_merged_before_outliers.shape)

Полных дубликатов событий: 2
Событий без записи о пользователе: 217
Размер объединённой таблицы: (179111, 6)


### 3.3. Поиск аномально активных пользователей

В анализе использовался порог **более 25 событий одного типа на пользователя**. Он заметно выше типичного уровня: медиана по основным событиям составляет примерно 11-12, а 99-й перцентиль - около 11-18.

Такие наблюдения помечаем как аномальные и исключаем из основного расчёта, но не объявляем автоматически ботами: высокая активность может быть реальной. Чтобы проверить, насколько вывод зависит от этого решения, ниже будет дополнительный анализ чувствительности **до и после** исключения аномальных пользователей.

In [ ]:
activity_before_outliers = df_merged_before_outliers[
    ~df_merged_before_outliers["event_type"].isin(RETURN_EVENTS)
].copy()

event_counts_by_user_type = (
    activity_before_outliers
    .groupby(["user_id", "event_type"])
    .size()
    .rename("event_count")
    .reset_index()
)

outlier_distribution = (
    event_counts_by_user_type
    .groupby("event_type")["event_count"]
    .describe(percentiles=[0.50, 0.90, 0.95, 0.99, 0.999])
)
display(outlier_distribution)

outlier_user_ids = (
    event_counts_by_user_type.loc[
        event_counts_by_user_type["event_count"] > OUTLIER_THRESHOLD,
        "user_id"
    ]
    .unique()
)

print(f"Пользователей с >{OUTLIER_THRESHOLD} событиями одного типа:", len(outlier_user_ids))

# Основная аналитическая выборка после очистки.
df_users_analysis = df_users_unique[
    ~df_users_unique["user_id"].isin(outlier_user_ids)
].copy()
df_merged_analysis = df_merged_before_outliers[
    ~df_merged_before_outliers["user_id"].isin(outlier_user_ids)
].copy()

print("Пользователей до исключения аномалий:", len(df_users_unique))
print("Пользователей после исключения аномалий:", len(df_users_analysis))
print("Исключено пользователей:", len(outlier_user_ids))
display(df_users_analysis["group"].value_counts().reindex(["A", "B"]))

,count,mean,std,min,50%,90%,95%,99%,99.9%,max
event_type,,,,,,,,,,
homework_completed,"5,886.0000",4.9207,11.3167,1.0000,4.0000,8.0000,9.0000,11.0000,246.9200,290.0000
homework_started,"5,992.0000",11.3863,11.4577,2.0000,11.0000,14.0000,15.0000,17.0000,258.0000,308.0000
login,"5,992.0000",12.5935,11.5277,3.0000,12.0000,15.0000,16.0000,18.0000,258.0540,291.0000


Пользователей с >25 событиями одного типа: 12
Пользователей до исключения аномалий: 5992
Пользователей после исключения аномалий: 5980
Исключено пользователей: 12


,count
group,
A,2992
B,2988


### 3.4. Проверка подозрительной автоматизированной активности

Для диагностики ботов анализируем интервалы между соседними событиями одного пользователя. Важно учитывать точность исходных временных меток: timestamp записан только до минуты, нельзя увидеть интервалы меньше 60 секунд (в большинстве случаев timestamp с 00 в секундах).

Минимальный положительный интервал - 60 секунд, нулевых интервалов нет. Это не даёт оснований объявлять оставшихся пользователей ботами, но и **не доказывает полного отсутствия автоматизации**: точности логов недостаточно для такой гарантии.

In [ ]:
activity_events = df_merged_analysis[
    ~df_merged_analysis["event_type"].isin(RETURN_EVENTS)
].copy()

activity_seq = activity_events.sort_values(["user_id", "timestamp"]).copy()
activity_seq["prev_timestamp"] = activity_seq.groupby("user_id")["timestamp"].shift(1)
activity_seq["gap_sec"] = (
    activity_seq["timestamp"] - activity_seq["prev_timestamp"]
).dt.total_seconds()

valid_gaps = activity_seq["gap_sec"].dropna()
print("Минимальный интервал, сек:", valid_gaps.min())
print("Нулевые интервалы:", (valid_gaps == 0).sum())
print("Отрицательные интервалы:", (valid_gaps < 0).sum())
print("Доля интервалов, кратных 60 секундам:", (valid_gaps % 60 == 0).mean())
print("Доля timestamp с second == 0:", (activity_events["timestamp"].dt.second == 0).mean())

gap_summary = valid_gaps.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])
display(gap_summary.to_frame("gap_sec"))

fig = px.histogram(
    valid_gaps[valid_gaps <= 600].to_frame("gap_sec"),
    x="gap_sec", nbins=60,
    title="Интервалы между соседними событиями до 10 минут",
    labels={"gap_sec": "Интервал, секунд", "count": "Число интервалов"}
)
fig.show()

Минимальный интервал, сек: 60.0
Нулевые интервалы: 0
Отрицательные интервалы: 0
Доля интервалов, кратных 60 секундам: 1.0
Доля timestamp с second == 0: 1.0


,gap_sec
count,"157,403.0000"
mean,"60,831.1450"
std,"93,092.7791"
min,60.0000
1%,60.0000
5%,120.0000
50%,"1,680.0000"
95%,"256,680.0000"
99%,"424,380.0000"
max,"1,121,100.0000"


### 3.5. Особенность событий Retention

Проверяем, как записаны маркеры возврата. Если timestamp совпадает у всех пользователей внутри типа события, его нельзя трактовать как индивидуальное время возвращения. Для анализа D7/D14 используем только наличие маркера у пользователя (`True`/`False`), а не его timestamp.

In [ ]:
retention_events_raw = df_merged_analysis[
    df_merged_analysis["event_type"].isin(RETURN_EVENTS)
].copy()

retention_timestamp_check = (
    retention_events_raw
    .groupby("event_type")
    .agg(
        events=("user_id", "size"),
        unique_users=("user_id", "nunique"),
        unique_timestamps=("timestamp", "nunique"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max")
    )
)
display(retention_timestamp_check)

,events,unique_users,unique_timestamps,first_timestamp,last_timestamp
event_type,,,,,
return_visit_d14,1945,1945,1,2026-05-18 18:00:00,2026-05-18 18:00:00
return_visit_d7,4505,4505,1,2026-05-11 18:00:00,2026-05-11 18:00:00


## 4. Sanity check: корректность сплитования

Проверяем три вещи:

1. После удаления конфликтующих `user_id` каждый пользователь находится только в одной группе.
2. Размеры A и B соответствуют ожидаемому сплиту 50/50.
3. Распределение учеников по классам сопоставимо между группами.

SRM проверяем после разрешения конфликтов назначения и отдельно показываем размеры выборок после фильтрации аномалий. Удаление пользователей по качеству логов не должно незаметно менять состав групп.

In [ ]:
def srm_check(user_table, label):
    sizes = user_table["group"].value_counts().reindex(["A", "B"], fill_value=0)
    observed = sizes.to_numpy()
    expected = np.repeat(observed.sum() / 2, 2)
    chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)
    print(f"{label}: A={observed[0]}, B={observed[1]}, всего={observed.sum()}")
    print(f"  Chi-square={chi2_stat:.4f}; p-value={p_value:.4f}")
    print("  Вывод:", "нет оснований отклонять сплит 50/50" if p_value >= ALPHA else "обнаружено SRM — требуется расследование")
    return sizes, chi2_stat, p_value

sizes_before_outliers, chi2_srm_before, p_srm_before = srm_check(
    df_users_unique, "После устранения неоднозначных назначений, до фильтрации аномалий"
)
sizes_after_outliers, chi2_srm_after, p_srm_after = srm_check(
    df_users_analysis, "Финальная аналитическая выборка"
)

# Проверка, что после очистки каждый user_id имеет ровно одну группу.
overlap_after_cleaning = (
    df_users_analysis.groupby("user_id")["group"].nunique().gt(1).sum()
)
print("Пользователей одновременно в A и B после очистки:", overlap_after_cleaning)

# Баланс по классам.
grade_group_table = pd.crosstab(df_users_analysis["grade"], df_users_analysis["group"])
grade_chi2, grade_p, grade_dof, grade_expected = stats.chi2_contingency(grade_group_table)
print("\nПроверка распределения по классам:")
print("Chi-square:", round(grade_chi2, 4), "p-value:", round(grade_p, 4), "df:", grade_dof)
display(grade_group_table)

После устранения неоднозначных назначений, до фильтрации аномалий: A=2997, B=2995, всего=5992
  Chi-square=0.0007; p-value=0.9794
  Вывод: нет оснований отклонять сплит 50/50
Финальная аналитическая выборка: A=2992, B=2988, всего=5980
  Chi-square=0.0027; p-value=0.9587
  Вывод: нет оснований отклонять сплит 50/50
Пользователей одновременно в A и B после очистки: 0

Проверка распределения по классам:
Chi-square: 8.0119 p-value: 0.2372 df: 6


group,A,B
grade,,
5,446,416
6,446,462
7,402,435
8,454,431
9,430,379
10,409,438
11,405,427


**Интерпретация sanity check.** В сохранённых результатах после очистки осталось 2 992 пользователя в A и 2 988 в B; для сплита 50/50 получено `p-value = 0.9587`. Пересечений между группами после исключения конфликтующих ID нет. Для распределения по классам получено `p-value = 0.2372`: статистически значимого дисбаланса не обнаружено. Эти проверки не доказывают идеальную рандомизацию по всем возможным признакам, но явного SRM и перекоса по классу не видно.

## 5. Агрегация событий на уровень пользователя

A/B-тест рандомизирует пользователей, а не отдельные события. Поэтому сначала превращаем логи в одну строку на пользователя, затем сравниваем метрики между группами.

Для каждого пользователя считаем:

- число `login`, `homework_started`, `homework_completed`;
- наличие маркеров Retention D7 и D14;
- долю завершённых домашних заданий среди начатых;
- наличие хотя бы одного завершённого задания;
- TTV - время от первого логина до первого завершённого задания.

In [ ]:
def build_user_metrics(user_table, event_table):
    """Собирает одну строку метрик на каждого пользователя из таблицы назначения и логов."""
    base = user_table[["user_id", "group", "grade"]].drop_duplicates("user_id").copy()

    activity = event_table[~event_table["event_type"].isin(RETURN_EVENTS)].copy()
    counts = (
        activity.groupby(["user_id", "event_type"]).size()
        .unstack(fill_value=0)
    )

    event_to_col = {
        "login": "login",
        "homework_started": "hw_started",
        "homework_completed": "completed_hw"
    }
    counts = counts.rename(columns=event_to_col)
    for col in ["login", "hw_started", "completed_hw"]:
        if col not in counts.columns:
            counts[col] = 0
    counts = counts[["login", "hw_started", "completed_hw"]]

    metrics = base.merge(counts, left_on="user_id", right_index=True, how="left")
    metrics[["login", "hw_started", "completed_hw"]] = (
        metrics[["login", "hw_started", "completed_hw"]].fillna(0).astype(int)
    )

    # Маркеры retention: если события нет, считаем False.
    ret_flags = event_table[
        event_table["event_type"].isin(RETURN_EVENTS)
    ].groupby(["user_id", "event_type"]).size().unstack(fill_value=0)

    for event_name, metric_name in [
        ("return_visit_d7", "retention_d7"),
        ("return_visit_d14", "retention_d14")
    ]:
        if event_name in ret_flags.columns:
            metrics[metric_name] = metrics["user_id"].map(ret_flags[event_name]).fillna(0).gt(0)
        else:
            metrics[metric_name] = False

    metrics["has_login"] = metrics["login"].gt(0)
    metrics["has_started"] = metrics["hw_started"].gt(0)
    metrics["has_completed"] = metrics["completed_hw"].gt(0)
    metrics["completion_rate"] = (
        metrics["completed_hw"] / metrics["hw_started"].replace(0, np.nan)
    )

    first_login = (
        event_table[event_table["event_type"] == "login"]
        .groupby("user_id")["timestamp"].min().rename("first_login")
    )
    first_completed = (
        event_table[event_table["event_type"] == "homework_completed"]
        .groupby("user_id")["timestamp"].min().rename("first_completed")
    )
    metrics = metrics.merge(first_login, on="user_id", how="left")
    metrics = metrics.merge(first_completed, on="user_id", how="left")
    metrics["ttv_hours"] = (
        metrics["first_completed"] - metrics["first_login"]
    ).dt.total_seconds() / 3600

    # Отрицательное TTV означает некорректную последовательность событий.
    metrics.loc[metrics["ttv_hours"] < 0, "ttv_hours"] = np.nan

    metrics["grade_segment"] = pd.cut(
        metrics["grade"],
        bins=[4, 7, 9, 11],
        labels=["5-7", "8-9", "10-11"]
    )
    return metrics

user_metrics = build_user_metrics(df_users_analysis, df_merged_analysis)
print("Размер user-level таблицы:", user_metrics.shape)
display(user_metrics.head())

Размер user-level таблицы: (5980, 16)


,user_id,group,grade,login,hw_started,completed_hw,retention_d7,retention_d14,has_login,has_started,has_completed,completion_rate,first_login,first_completed,ttv_hours,grade_segment
0,U300000,B,5,14,11,6,True,True,True,True,True,0.5455,2026-05-04 19:36:00,2026-05-11 19:14:00,167.6333,5-7
1,U300001,B,9,14,11,3,True,True,True,True,True,0.2727,2026-05-04 20:12:00,2026-05-12 17:09:00,188.9500,8-9
2,U300002,B,9,12,11,4,True,True,True,True,True,0.3636,2026-05-06 18:47:00,2026-05-10 19:51:00,97.0667,8-9
3,U300003,A,5,10,10,3,True,True,True,True,True,0.3000,2026-05-04 19:43:00,2026-05-09 17:18:00,117.5833,5-7
4,U300004,B,11,13,12,3,True,False,True,True,True,0.2500,2026-05-04 19:09:00,2026-05-04 19:58:00,0.8167,10-11


### 5.1. Проверка получившихся метрик

Проверим, не потеряли ли пользователей при сборке таблицы, все ли пользователи имеют хотя бы один логин и старт домашнего задания, а также сколько наблюдений доступно для TTV.

In [ ]:
print("Уникальных пользователей в таблице:", user_metrics["user_id"].nunique())
print("Пользователей в исходной очищенной таблице:", len(df_users_analysis))
print("Пользователей с логином:", user_metrics["has_login"].sum())
print("Пользователей, начавших хотя бы одно ДЗ:", user_metrics["has_started"].sum())
print("Пользователей, завершивших хотя бы одно ДЗ:", user_metrics["has_completed"].sum())
print("Пользователей с рассчитанным TTV:", user_metrics["ttv_hours"].notna().sum())
print("Отрицательное TTV после очистки:", user_metrics["ttv_hours"].lt(0).sum())

user_metrics.groupby("group").agg(
    users=("user_id", "nunique"),
    retention_d7=("retention_d7", "mean"),
    retention_d14=("retention_d14", "mean"),
    completion_rate=("completion_rate", "mean"),
    completed_hw_per_user=("completed_hw", "mean"),
    hw_started_per_user=("hw_started", "mean"),
    login_per_user=("login", "mean"),
    ttv_hours=("ttv_hours", "mean")
)

Уникальных пользователей в таблице: 5980
Пользователей в исходной очищенной таблице: 5980
Пользователей с логином: 5980
Пользователей, начавших хотя бы одно ДЗ: 5980
Пользователей, завершивших хотя бы одно ДЗ: 5874
Пользователей с рассчитанным TTV: 5874
Отрицательное TTV после очистки: 0


,users,retention_d7,retention_d14,completion_rate,completed_hw_per_user,hw_started_per_user,login_per_user,ttv_hours
group,,,,,,,,
A,2992,0.7269,0.2881,0.3461,3.5802,10.3606,11.5114,88.7664
B,2988,0.7798,0.3624,0.4380,5.1068,11.4153,12.6714,64.3761


### 5.2. Анализ чувствительности: что меняется без удаления аномальных пользователей

Порог 25 событий - практическое правило, а не абсолютное доказательство ботов. Поэтому повторно собираем пользовательские метрики на данных **до** удаления 12 аномальных пользователей. Если бизнес-вывод резко меняется, этому нужно уделить внимание.

In [ ]:
user_metrics_with_outliers = build_user_metrics(df_users_unique, df_merged_before_outliers)

sensitivity_metrics = [
    "retention_d7", "retention_d14", "completion_rate",
    "completed_hw", "hw_started", "login", "ttv_hours"
]

sensitivity_rows = []
for metric in sensitivity_metrics:
    for group in ["A", "B"]:
        primary_mean = user_metrics.loc[user_metrics["group"] == group, metric].mean()
        with_outliers_mean = user_metrics_with_outliers.loc[
            user_metrics_with_outliers["group"] == group, metric
        ].mean()
        sensitivity_rows.append({
            "metric": metric,
            "group": group,
            "without_anomalies": primary_mean,
            "with_anomalies": with_outliers_mean,
            "difference": with_outliers_mean - primary_mean
        })

sensitivity_table = pd.DataFrame(sensitivity_rows)
display(sensitivity_table)

,metric,group,without_anomalies,with_anomalies,difference
0,retention_d7,A,0.7269,0.7271,0.0001
1,retention_d7,B,0.7798,0.7796,-0.0002
2,retention_d14,A,0.2881,0.2876,-0.0005
3,retention_d14,B,0.3624,0.3623,-0.0002
4,completion_rate,A,0.3461,0.3471,0.0010
5,completion_rate,B,0.4380,0.4392,0.0012
6,completed_hw,A,3.5802,3.9690,0.3888
7,completed_hw,B,5.1068,5.6988,0.5921
8,hw_started,A,10.3606,10.7591,0.3985
9,hw_started,B,11.4153,12.0140,0.5987


Не вявлено сильных изменений, все корректно.

## 6. Воронка и разведочный анализ

### 6.1. Конверсионная воронка

Для каждого шага используем уникальных пользователей, а не число событий. Воронка отражает долю пользователей, у которых вообще было событие данного типа:

`login → homework_started → homework_completed`.

У этой воронки есть ограничение: по самим логам нельзя подтвердить, что `homework_completed` означает полное завершение модуля или курса.

In [ ]:
funnel_summary = (
    user_metrics.groupby("group")
    .agg(
        users=("user_id", "nunique"),
        login_users=("has_login", "sum"),
        started_users=("has_started", "sum"),
        completed_users=("has_completed", "sum")
    )
    .reindex(["A", "B"])
)

funnel_summary["login_conversion"] = funnel_summary["login_users"] / funnel_summary["users"]
funnel_summary["started_from_login"] = funnel_summary["started_users"] / funnel_summary["login_users"]
funnel_summary["completed_from_login"] = funnel_summary["completed_users"] / funnel_summary["login_users"]
display(funnel_summary)

funnel_long = funnel_summary.reset_index().melt(
    id_vars="group",
    value_vars=["login_users", "started_users", "completed_users"],
    var_name="step", value_name="count"
)

funnel_labels = {
    "login_users": "Login",
    "started_users": "Начал ДЗ",
    "completed_users": "Завершил ≥1 ДЗ"
}
funnel_long["step"] = funnel_long["step"].map(funnel_labels)

fig = px.bar(
    funnel_long, x="step", y="count", color="group", barmode="group",
    category_orders={"step": ["Login", "Начал ДЗ", "Завершил ≥1 ДЗ"]},
    title="Воронка пользовательской активности",
    labels={"step": "Шаг воронки", "count": "Уникальные пользователи", "group": "Группа"}
)
fig.show()

,users,login_users,started_users,completed_users,login_conversion,started_from_login,completed_from_login
group,,,,,,,
A,2992,2992,2992,2921,1.0000,1.0000,0.9763
B,2988,2988,2988,2953,1.0000,1.0000,0.9883


### 6.2. Распределения метрик

Сравниваем распределения, а не только средние. Время до первого завершения и количество событий обычно имеют правостороннюю асимметрию, поэтому дополнительно смотрим медианы и используем непараметрический критерий Манна-Уитни как проверку устойчивости результата.

In [ ]:
for metric, title, y_label in [
    ("completion_rate", "Распределение пользовательского completion rate", "completed / started"),
    ("ttv_hours", "Распределение Time to Value", "Часы до первого completed"),
    ("completed_hw", "Число завершённых домашних заданий на пользователя", "Количество ДЗ"),
]:
    fig = px.box(
        user_metrics, x="group", y=metric, color="group", points="outliers",
        title=title, labels={"group": "Группа", metric: y_label}
    )
    fig.show()

Даже визуально видны положительные изменения для тестовой группы относительно контрольной. Completion rate и число выполненных дз на пользователей больше для группы B. TTV же наоборот меньше для тестовой группы.

## 7. Статистические тесты и доверительные интервалы

### 7.1. Сравнение долей

Для бинарных показателей используем z-тест двух независимых долей. Доверительный интервал для разницы долей рассчитываем методом Newcombe на основе интервалов Wilson - он устойчивее простого Wald-интервала.

Знак эффекта везде одинаковый: `B − A`.

In [ ]:
def test_binary_metric(data, metric, metric_label=None):
    a = data.loc[data["group"] == "A", metric].dropna().astype(int)
    b = data.loc[data["group"] == "B", metric].dropna().astype(int)

    count_a, count_b = int(a.sum()), int(b.sum())
    n_a, n_b = len(a), len(b)
    p_a, p_b = count_a / n_a, count_b / n_b

    z_stat, p_value = proportions_ztest(
        count=[count_b, count_a],
        nobs=[n_b, n_a],
        alternative="two-sided"
    )
    ci_low, ci_high = confint_proportions_2indep(
        count1=count_b, nobs1=n_b,
        count2=count_a, nobs2=n_a,
        method="newcomb"
    )

    difference = p_b - p_a
    relative_lift = difference / p_a if p_a else np.nan

    return {
        "metric": metric_label or metric,
        "metric_code": metric,
        "test": "z-test двух долей",
        "n_A": n_a,
        "n_B": n_b,
        "value_A": p_a,
        "value_B": p_b,
        "difference_B_minus_A": difference,
        "relative_lift": relative_lift,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": p_value,
        "p_mann_whitney": np.nan
    }

binary_results = [
    test_binary_metric(user_metrics, "retention_d7", "Retention D7"),
    test_binary_metric(user_metrics, "retention_d14", "Retention D14"),
    test_binary_metric(user_metrics, "has_completed", "Завершил хотя бы одно ДЗ")
]

### 7.2. Сравнение непрерывных метрик

Для непрерывных метрик используем t-тест Уэлча: он не требует равенства дисперсий в группах. Дополнительно считаем U-критерий Манна - Уитни. Для разницы средних строим percentile bootstrap CI: много раз случайно выбираем пользователей с возвращением внутри каждой группы и рассчитываем разницу средних B - A.


In [ ]:
def bootstrap_mean_difference(a, b, n_boot=N_BOOT, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    bootstrap_diffs = np.empty(n_boot)

    for i in range(n_boot):
        a_sample = rng.choice(a, size=len(a), replace=True)
        b_sample = rng.choice(b, size=len(b), replace=True)
        bootstrap_diffs[i] = b_sample.mean() - a_sample.mean()

    return np.quantile(bootstrap_diffs, [0.025, 0.975])


def test_continuous_metric(data, metric, metric_label=None):
    a = data.loc[data["group"] == "A", metric].dropna().astype(float).to_numpy()
    b = data.loc[data["group"] == "B", metric].dropna().astype(float).to_numpy()

    welch_stat, welch_p = stats.ttest_ind(b, a, equal_var=False, nan_policy="omit")
    mw_stat, mw_p = stats.mannwhitneyu(b, a, alternative="two-sided")
    ci_low, ci_high = bootstrap_mean_difference(a, b)

    mean_a, mean_b = a.mean(), b.mean()
    difference = mean_b - mean_a
    relative_lift = difference / mean_a if mean_a else np.nan

    return {
        "metric": metric_label or metric,
        "metric_code": metric,
        "test": "t-test Уэлча",
        "n_A": len(a),
        "n_B": len(b),
        "value_A": mean_a,
        "value_B": mean_b,
        "difference_B_minus_A": difference,
        "relative_lift": relative_lift,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": welch_p,
        "p_mann_whitney": mw_p
    }

continuous_results = [
    test_continuous_metric(user_metrics, "completion_rate", "Средний пользовательский completion rate"),
    test_continuous_metric(user_metrics, "completed_hw", "Завершённые ДЗ на пользователя"),
    test_continuous_metric(user_metrics, "hw_started", "Начатые ДЗ на пользователя"),
    test_continuous_metric(user_metrics, "login", "Логины на пользователя"),
    test_continuous_metric(user_metrics, "ttv_hours", "TTV среди завершивших хотя бы одно ДЗ")
]

### 7.3. Сводная таблица статистических результатов

В качестве основного p-value берём z-тест для долей и t-тест Уэлча для непрерывных метрик. Mann–Whitney оставляем как дополнительную проверку для скошенных распределений. Поправку Холма применяем к набору основных проверок, чтобы учесть множественные сравнения.

In [ ]:
results = pd.DataFrame(binary_results + continuous_results)
results["p_value_holm"] = multipletests(results["p_value"], alpha=ALPHA, method="holm")[1]

display_cols = [
    "metric", "test", "n_A", "n_B", "value_A", "value_B",
    "difference_B_minus_A", "relative_lift", "ci_low", "ci_high",
    "p_value", "p_value_holm", "p_mann_whitney"
]
display(results[display_cols].sort_values("p_value"))

def format_p_value(value):
    if pd.isna(value):
        return "—"
    if value < 0.001:
        return "< 0.001"
    return f"{value:.3f}"

results_for_report = results.copy()
results_for_report["p_value_text"] = results_for_report["p_value"].apply(format_p_value)
results_for_report["p_value_holm_text"] = results_for_report["p_value_holm"].apply(format_p_value)
display(results_for_report[["metric", "difference_B_minus_A", "ci_low", "ci_high", "p_value_text", "p_value_holm_text"]])

,metric,test,n_A,n_B,value_A,value_B,difference_B_minus_A,relative_lift,ci_low,ci_high,p_value,p_value_holm,p_mann_whitney
4,Завершённые ДЗ на пользователя,t-test Уэлча,2992,2988,3.5802,5.1068,1.5265,0.4264,1.4182,1.6402,0.0000,0.0000,0.0000
3,Средний пользовательский completion rate,t-test Уэлча,2992,2988,0.3461,0.4380,0.0919,0.2656,0.0834,0.1005,0.0000,0.0000,0.0000
6,Логины на пользователя,t-test Уэлча,2992,2988,11.5114,12.6714,1.1600,0.1008,1.0319,1.2874,0.0000,0.0000,0.0000
5,Начатые ДЗ на пользователя,t-test Уэлча,2992,2988,10.3606,11.4153,1.0547,0.1018,0.9280,1.1828,0.0000,0.0000,0.0000
7,TTV среди завершивших хотя бы одно ДЗ,t-test Уэлча,2921,2953,88.7664,64.3761,-24.3903,-0.2748,-29.4008,-19.5713,0.0000,0.0000,0.0000
1,Retention D14,z-test двух долей,2992,2988,0.2881,0.3624,0.0743,0.2581,0.0506,0.0980,0.0000,0.0000,NaN
0,Retention D7,z-test двух долей,2992,2988,0.7269,0.7798,0.0528,0.0727,0.0310,0.0746,0.0000,0.0000,NaN
2,Завершил хотя бы одно ДЗ,z-test двух долей,2992,2988,0.9763,0.9883,0.0120,0.0123,0.0054,0.0189,0.0004,0.0004,NaN


,metric,difference_B_minus_A,ci_low,ci_high,p_value_text,p_value_holm_text
0,Retention D7,0.0528,0.0310,0.0746,< 0.001,< 0.001
1,Retention D14,0.0743,0.0506,0.0980,< 0.001,< 0.001
2,Завершил хотя бы одно ДЗ,0.0120,0.0054,0.0189,< 0.001,< 0.001
3,Средний пользовательский completion rate,0.0919,0.0834,0.1005,< 0.001,< 0.001
4,Завершённые ДЗ на пользователя,1.5265,1.4182,1.6402,< 0.001,< 0.001
5,Начатые ДЗ на пользователя,1.0547,0.9280,1.1828,< 0.001,< 0.001
6,Логины на пользователя,1.1600,1.0319,1.2874,< 0.001,< 0.001
7,TTV среди завершивших хотя бы одно ДЗ,-24.3903,-29.4008,-19.5713,< 0.001,< 0.001


Судя по всем статистическим проверкам, была выявлена статистическая значимость по всем метрикам. Нулевая гипотеза об отстутствии стат. значимости отвергается.

## 8. MDE и статистическая мощность

**MDE (Minimum Detectable Effect)** - минимальная разница, которую эксперимент заданного размера способен обнаружить при выбранных `alpha` и мощности. Здесь рассчитываем MDE при мощности 80% и `alpha = 0.05`.

Не используем post-hoc observed power как аргумент в пользу результата: она во многом повторяет информацию p-value. Вместо этого показываем, какой эффект выборка могла надёжно обнаружить.

In [ ]:
def proportion_mde(p_baseline, n_a, n_b, alpha=ALPHA, target_power=0.80):
    effect_h = NormalIndPower().solve_power(
        effect_size=None,
        nobs1=n_a,
        alpha=alpha,
        power=target_power,
        ratio=n_b / n_a,
        alternative="two-sided"
    )
    # Cohen's h = 2*asin(sqrt(p_B)) - 2*asin(sqrt(p_A))
    p_detectable = np.sin(np.arcsin(np.sqrt(p_baseline)) + effect_h / 2) ** 2
    return abs(p_detectable - p_baseline), effect_h

mde_rows = []
for metric, label in [
    ("retention_d7", "Retention D7"),
    ("retention_d14", "Retention D14"),
    ("has_completed", "Завершил хотя бы одно ДЗ")
]:
    a = user_metrics.loc[user_metrics["group"] == "A", metric].astype(int)
    b = user_metrics.loc[user_metrics["group"] == "B", metric].astype(int)
    mde_abs, effect_h = proportion_mde(a.mean(), len(a), len(b))
    observed_effect = b.mean() - a.mean()
    mde_rows.append({
        "metric": label,
        "MDE_absolute": mde_abs,
        "MDE_percentage_points": mde_abs * 100,
        "observed_difference": observed_effect,
        "observed_difference_pp": observed_effect * 100,
        "Cohens_h_MDE": effect_h
    })

# Отдельный MDE для TTV. Расчёт относится только к пользователям с известным TTV.
a_ttv = user_metrics.loc[user_metrics["group"] == "A", "ttv_hours"].dropna()
b_ttv = user_metrics.loc[user_metrics["group"] == "B", "ttv_hours"].dropna()
pooled_sd = np.sqrt(
    ((len(a_ttv) - 1) * a_ttv.var(ddof=1) + (len(b_ttv) - 1) * b_ttv.var(ddof=1))
    / (len(a_ttv) + len(b_ttv) - 2)
)
mde_d = TTestIndPower().solve_power(
    effect_size=None,
    nobs1=len(a_ttv),
    ratio=len(b_ttv) / len(a_ttv),
    alpha=ALPHA,
    power=0.80,
    alternative="two-sided"
)
mde_rows.append({
    "metric": "TTV среди завершивших хотя бы одно ДЗ",
    "MDE_absolute": mde_d * pooled_sd,
    "MDE_percentage_points": np.nan,
    "observed_difference": b_ttv.mean() - a_ttv.mean(),
    "observed_difference_pp": np.nan,
    "Cohens_h_MDE": mde_d
})

mde_table = pd.DataFrame(mde_rows)
display(mde_table)

,metric,MDE_absolute,MDE_percentage_points,observed_difference,observed_difference_pp,Cohens_h_MDE
0,Retention D7,0.0317,3.1658,0.0528,5.2847,0.0725
1,Retention D14,0.0333,3.3342,0.0743,7.4348,0.0725
2,Завершил хотя бы одно ДЗ,0.0098,0.9769,0.0120,1.2016,0.0725
3,TTV среди завершивших хотя бы одно ДЗ,7.0877,NaN,-24.3903,NaN,0.0731


Для D7 в сохранённых расчётах MDE составляет около **3,17 п.п.**, а наблюдаемая разница — **5,28 п.п.** Для TTV MDE — около **7,09 часа**, наблюдаемое снижение — около **24,39 часа** (среди пользователей с рассчитанным TTV).

## 9. Сегментный анализ по классам

Общий средний эффект может скрывать разное поведение отдельных групп. Разбиваем учеников на сегменты:

- 5–7 классы;
- 8–9 классы;
- 10–11 классы.

Сначала сравниваем значения метрик описательно. Затем проверяем, отличается ли эффект тестовой группы между сегментами с помощью модели с взаимодействием `group × grade_segment`. Для бинарных D7/D14 используем логистическую регрессию и likelihood-ratio test взаимодействия.

In [ ]:
segment_summary = (
    user_metrics
    .groupby(["grade_segment", "group"], observed=True)
    .agg(
        users=("user_id", "nunique"),
        retention_d7=("retention_d7", "mean"),
        retention_d14=("retention_d14", "mean"),
        completion_rate=("completion_rate", "mean"),
        completed_hw=("completed_hw", "mean"),
        ttv_hours=("ttv_hours", "mean")
    )
    .reset_index()
)
display(segment_summary)

segment_plot = segment_summary.melt(
    id_vars=["grade_segment", "group", "users"],
    value_vars=["retention_d7", "retention_d14", "completion_rate"],
    var_name="metric", value_name="rate"
)
segment_plot["rate_percent"] = segment_plot["rate"] * 100

fig = px.bar(
    segment_plot, x="grade_segment", y="rate_percent", color="group",
    barmode="group", facet_col="metric",
    title="Retention и пользовательский completion rate по сегментам",
    labels={"grade_segment": "Классы", "rate_percent": "%", "group": "Группа", "metric": "Метрика"}
)
fig.show()

,grade_segment,group,users,retention_d7,retention_d14,completion_rate,completed_hw,ttv_hours
0,5-7,A,1294,0.7311,0.2828,0.3481,3.5951,87.0457
1,5-7,B,1313,0.8462,0.4448,0.5330,6.6603,39.8786
2,8-9,A,884,0.7262,0.2907,0.3417,3.5249,91.3127
3,8-9,B,810,0.7790,0.3457,0.4334,4.8815,60.7184
4,10-11,A,814,0.7211,0.2936,0.3476,3.6167,88.7308
5,10-11,B,865,0.6798,0.2532,0.2980,2.9595,106.4785


In [ ]:
GRADES = [5, 6, 7, 8, 9, 10, 11]

for metric, title, ylabel in [
    ('retention_d7',    'Retention D7 по классам',    'Retention D7, %'),
    ('retention_d14',   'Retention D14 по классам',   'Retention D14, %'),
    ('completion_rate', 'Completion Rate по классам', 'Completion Rate, %'),
]:
    fig = px.bar(
        segment_summary,
        x='grade_segment',
        y=metric,
        color='group',
        barmode='group',
        text_auto='.1%',
        color_discrete_map={'A': '#636efa', 'B': '#EF553B'},
        labels={'grade': 'Класс', metric: ylabel},
        title=title,
    )
    fig.update_xaxes(type='category')
    fig.update_yaxes(tickformat='.0%')
    fig.update_traces(textposition='outside')
    fig.show()

In [ ]:
def binary_interaction_test(data, outcome):
    model_data = data[[outcome, "group", "grade_segment"]].dropna().copy()
    model_data[outcome] = model_data[outcome].astype(int)
    model_data["group"] = pd.Categorical(model_data["group"], categories=["A", "B"])
    model_data["grade_segment"] = pd.Categorical(
        model_data["grade_segment"], categories=["5-7", "8-9", "10-11"]
    )

    full_model = smf.glm(
        f"{outcome} ~ C(group) * C(grade_segment)",
        data=model_data,
        family=sm.families.Binomial()
    ).fit()
    reduced_model = smf.glm(
        f"{outcome} ~ C(group) + C(grade_segment)",
        data=model_data,
        family=sm.families.Binomial()
    ).fit()

    lr_stat = 2 * (full_model.llf - reduced_model.llf)
    df_diff = int(full_model.df_model - reduced_model.df_model)
    p_interaction = stats.chi2.sf(lr_stat, df_diff)
    return {
        "metric": outcome,
        "LR_statistic": lr_stat,
        "df": df_diff,
        "p_interaction": p_interaction
    }, full_model

interaction_rows = []
interaction_models = {}
for metric in ["retention_d7", "retention_d14", "has_completed"]:
    result, fitted_model = binary_interaction_test(user_metrics, metric)
    interaction_rows.append(result)
    interaction_models[metric] = fitted_model

interaction_results = pd.DataFrame(interaction_rows)
interaction_results["p_interaction_holm"] = multipletests(
    interaction_results["p_interaction"], method="holm"
)[1]
display(interaction_results)

,metric,LR_statistic,df,p_interaction,p_interaction_holm
0,retention_d7,38.8353,2,0.0000,0.0000
1,retention_d14,45.1865,2,0.0000,0.0000
2,has_completed,44.7109,2,0.0000,0.0000


### Интерпретация сегментов по сохранённым значениям исходного ноутбука

| Сегмент | Retention D7: A → B | Retention D14: A → B | Completion rate: A → B |
|---|---:|---:|---:|
| 5–7 классы | 73,11% → 84,62% (**+11,51 п.п.**) | 28,28% → 44,48% (**+16,19 п.п.**) | 34,81% → 53,30% (**+18,49 п.п.**) |
| 8–9 классы | 72,62% → 77,90% (**+5,28 п.п.**) | 29,07% → 34,57% (**+5,50 п.п.**) | 34,17% → 43,34% (**+9,17 п.п.**) |
| 10–11 классы | 72,11% → 67,98% (**−4,14 п.п.**) | 29,36% → 25,32% (**−4,04 п.п.**) | 34,76% → 29,80% (**−4,96 п.п.**) |

Эффект положительный у учеников 5–9 классов, но отрицательный у 10–11 классов. Это важный сигнал неоднородности эффекта. Не следует автоматически называть его парадоксом Симпсона: распределение по классам между A и B статистически значимо не различалось (`p = 0.2372`), а основное наблюдение здесь — именно различная реакция сегментов на механику.

Likelihood-ratio тест взаимодействия `group × grade_segment` даёт ориентировочно `p ≈ 3,7 · 10⁻⁹` для D7 и `p ≈ 1,5 · 10⁻¹⁰` для D14.

## 10. Бизнес-интерпретация и решение

### Что говорят результаты в среднем по выборке

- Retention D7 вырос на **5,28 п.п.** (с 72,69% до 77,98%; исходный z-тест: `p < 0.001`).
- Retention D14 вырос на **7,43 п.п.** (с 28,81% до 36,24%; исходный z-тест: `p < 0.001`).
- Средний пользовательский коэффициент `completed_hw / hw_started` вырос на **9,19 п.п.** (с 34,61% до 43,80%; Welch и Mann–Whitney: `p < 0.001`).
- Среднее количество завершённых домашних заданий на пользователя выросло с **3,58 до 5,11** (+42,6%).
- TTV снизился с **88,77 до 64,38 часа** (−27,5%), однако этот вывод относится только к пользователям, которые завершили хотя бы одно домашнее задание.

### Что мешает безусловному запуску

- У учеников 10–11 классов Retention D7, D14 и средний коэффициент завершения домашних заданий снизились в тестовой группе.
- В данных нет прямой метрики полного завершения курса/модуля, поэтому центральная гипотеза о росте Course Completion Rate проверена только приближенными метриками.
- Маркеры D7/D14 нельзя перепроверить по timestamp, поскольку временные метки синтетические/одинаковые для всех пользователей данного типа.


### Рекомендация

**Не раскатывать новую механику сразу на 100% аудитории.** Предварительные данные поддерживают запуск для учеников 5–9 классов, где все ключевые продуктовые показатели улучшаются. Для 10–11 классов механику следует отправить на доработку: возможно, текущие достижения недостаточно релевантны для старших учеников или мешают их привычному сценарию обучения.

Следующий шаг - подтвердить эффект на повторном эксперименте с заранее зафиксированными primary-метриками, корректным событием полного прохождения модуля и проверяемым определением D7/D14.

## 11. Ограничения и что улучшить в следующем эксперименте

1. **Фактическое завершение курса.** Добавить отдельное событие `module_completed` или `course_completed`, чтобы проверить исходную гипотезу напрямую, а не через отношение событий домашних заданий.
2. **Retention.** Хранить фактическое время пользовательского возврата и дату первого контакта/назначения в тест. Тогда D7/D14 можно будет пересчитать из событий и корректно учесть пользователей, у которых ещё не прошло полное окно наблюдения.
3. **TTV.** Считать время от первого логина до первого завершения на уровне всей рандомизированной выборки и дополнительно показывать долю пользователей без завершения. Сейчас сравнивается только подвыборка завершивших.
4. **Боты и аномалии.** Добавить более точные timestamps и технические признаки (user-agent, source, session_id, event_id).


## 12. Краткий итог для стейкхолдеров

В среднем по эксперименту геймификация связана с ростом возвратов на 7-й и 14-й день и более высокой активностью по домашним заданиям. Эффект неоднороден: у учеников 5-9 классов показатели улучшаются, а у учеников 10-11 классов - снижаются. Поэтому разумное решение - **не делать общий rollout**, а провести ограниченный запуск для младших/средних классов и отдельно переработать сценарий для старших, одновременно улучшив измерение завершения курса и retention.

---

## Приложение. Словарь метрик

| Метрика | Определение в этом проекте | Как трактовать |
|---|---|---|
| Retention D7 | Доля пользователей с событием `return_visit_d7` | Возврат на 7-й день по готовому маркеру |
| Retention D14 | Доля пользователей с событием `return_visit_d14` | Возврат на 14-й день по готовому маркеру |
| Completion incidence | Доля пользователей хотя бы с одним `homework_completed` | Дошёл до завершения хотя бы одного ДЗ |
| User-level completion rate | Для пользователя `completed_hw / hw_started`, затем среднее по группе | Средняя доля завершённых начатых ДЗ на пользователя |
| TTV | Время от первого `login` до первого `homework_completed` | Сколько времени прошло до первого завершённого ДЗ; только среди завершивших |
| SRM | Chi-square goodness-of-fit для фактического сплита против 50/50 | Проверка нарушения ожидаемого распределения пользователей |
| MDE | Минимальный различимый эффект при alpha 5% и мощности 80% | Чувствительность выборки, а не оценка реального эффекта |